In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    # Gemma 4 requires transformers >= 5.5.0 — do NOT pin to 4.x here
    !pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
# Gemma 4 requires transformers >= 5.5.0
!pip install --upgrade --no-deps "transformers>=5.5.0" tokenizers "trl>=0.28.0" unsloth unsloth_zoo

In [3]:
%%capture
!pip install --no-deps --upgrade timm

In [4]:
from unsloth import FastVisionModel
import torch
# max_seq_length = 4096 # Can increase for longer reasoning traces
# lora_rank = 32 # Larger rank = smarter, but slower

gemma4_models = [
    # Gemma-4 instruct models:
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-E4B-it",
    "unsloth/gemma-4-31B-it",
    "unsloth/gemma-4-26B-A4B-it",
    # Gemma-4 base models:
    "unsloth/gemma-4-E2B",
    "unsloth/gemma-4-E4B",
    "unsloth/gemma-4-31B",
    "unsloth/gemma-4-26B-A4B",
] # More models at https://huggingface.co/unsloth

model, processor = FastVisionModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    # max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    # fast_inference = False,
    use_gradient_checkpointing="unsloth",
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.6: Fast Gemma4 patching. Transformers: 5.5.4.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

In [5]:
model

Gemma4ForConditionalGeneration(
  (model): Gemma4Model(
    (language_model): Gemma4TextModel(
      (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 1536, padding_idx=0)
      (layers): ModuleList(
        (0-3): 4 x Gemma4TextDecoderLayer(
          (self_attn): Gemma4TextAttention(
            (q_proj): Linear(in_features=1536, out_features=2048, bias=False)
            (q_norm): Gemma4RMSNorm()
            (k_norm): Gemma4RMSNorm()
            (v_norm): Gemma4RMSNorm()
            (k_proj): Linear(in_features=1536, out_features=256, bias=False)
            (v_proj): Linear(in_features=1536, out_features=256, bias=False)
            (o_proj): Linear(in_features=2048, out_features=1536, bias=False)
          )
          (mlp): Gemma4TextMLP(
            (gate_proj): Linear(in_features=1536, out_features=6144, bias=False)
            (up_proj): Linear(in_features=1536, out_features=6144, bias=False)
            (down_proj): Linear(in_features=6144, out_features=1536, bias=False)


In [6]:
model=FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
    target_modules="all-linear"
    
    
)

In [7]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma4ForConditionalGeneration(
      (model): Gemma4Model(
        (language_model): Gemma4TextModel(
          (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 1536, padding_idx=0)
          (layers): ModuleList(
            (0-3): 4 x Gemma4TextDecoderLayer(
              (self_attn): Gemma4TextAttention(
                (q_proj): lora.Linear(
                  (base_layer): Linear(in_features=1536, out_features=2048, bias=False)
                  (lora_dropout): ModuleDict(
                    (default): Identity()
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=1536, out_features=16, bias=False)
                  )
                  (lora_B): ModuleDict(
                    (default): Linear(in_features=16, out_features=2048, bias=False)
                  )
                  (lora_embedding_A): ParameterDict()
                  (lora_embedding_B): Parame

In [8]:
from datasets import load_dataset
dataset=load_dataset("unsloth/LaTeX_OCR",split="train")

In [9]:
from IPython.display import display,Math,Latex
latex=dataset[2]["text"]
display(Math(latex))

<IPython.core.display.Math object>

In [10]:
instruct="Write the Latex Representation for this image"
def convert_to_converstion(sample):
    conversion=[
        {
            "role":"user",
            "content":[
                {"type":"text","text":instruct},
                {"type":"image","image":sample["image"]},
            ],
        },
            {"role":"assistant","content":[{"type":"text","text":sample["text"]}]},
    ]
    return {"messages":conversion}


In [11]:
convert_dataset=[convert_to_converstion(sample) for sample in dataset]

In [12]:
convert_dataset[0]

{'messages': [{'role': 'user',
   'content': [{'type': 'text',
     'text': 'Write the Latex Representation for this image'},
    {'type': 'image',
     'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=160x40>}]},
  {'role': 'assistant',
   'content': [{'type': 'text',
     'text': '{ \\frac { N } { M } } \\in { \\bf Z } , { \\frac { M } { P } } \\in { \\bf Z } , { \\frac { P } { Q } } \\in { \\bf Z }'}]}]}

In [13]:
from unsloth import get_chat_template
processor=get_chat_template(
    processor,
    "gemma-4"
)

In [14]:
image=dataset[2]['image']
instruction = "Write the LaTeX representation for this image."
messages=[
    {
        "role":"user",
        "content":[{"type":"image"},{"type":"text","text":instruction}],
        
    } 

]
input_text=processor.apply_chat_template(messages, add_generation_prompt = True)
inputs=processor(
        image,
        input_text,
        add_special_tokens=False,
        return_tensors="pt",
        
    ).to("cuda")

from transformers import TextStreamer
text_streamer=TextStreamer(processor,skip_prompt=True)
result = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                        use_cache = True, temperature = 0.7, top_p = 0.95, top_k = 64)

```latex
H' = \beta N \int d\lambda \left\{ \frac{1}{2\beta^2 N^2} \partial_\lambda \xi^\dagger \partial_\lambda \xi + V(\lambda) \xi^\dagger \xi \right\}.
```<turn|>


In [15]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer,SFTConfig
trainer=SFTTrainer(
    model=model,
    train_dataset=convert_dataset,
    processing_class=processor.tokenizer,
    data_collator=UnslothVisionDataCollator(model,processor),
    args=SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        max_grad_norm = 0.3,
        warmup_ratio = 0.03,
        max_steps = 60,
        # num_train_epochs = 5, # Set this instead of max_steps for full training runs
        learning_rate = 2e-4,
        logging_steps = 1,
        save_strategy = "steps",
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # For Weights and Biases or others

        # You MUST put the below items for vision finetuning:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
    )
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Model does not have a default image size - using 512


In [16]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
10.156 GB of memory reserved.


In [17]:
traind_status=trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 68,686 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 28,680,192 of 5,132,977,696 (0.56% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Step,Training Loss
1,3.296473
2,3.621904
3,3.503518
4,4.662590
5,3.723402
6,3.519402
7,2.407836
8,2.075284
9,2.190318
10,1.050351


In [20]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{traind_status.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(traind_status.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

286.9914 seconds used for training.
4.78 minutes used for training.
Peak reserved memory = 10.498 GB.
Peak reserved memory for training = 0.342 GB.
Peak reserved memory % of max memory = 72.087 %.
Peak reserved memory for training % of max memory = 2.348 %.


In [21]:
image = dataset[10]["image"]
instruction = "Write the LaTeX representation for this image."

messages = [
    {
        "role": "user",
        "content": [{"type": "image"}, {"type": "text", "text": instruction}],
    }
]

input_text = processor.apply_chat_template(messages, add_generation_prompt = True)

inputs = processor(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer

text_streamer = TextStreamer(processor, skip_prompt = True)
result = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                        use_cache = True, temperature = 1.0, top_p = 0.95, top_k = 64)

\left[ \begin{array}{c} { \left[ B _ { n } ^ { \dagger } , b _ { 2 } ^ { \dagger } \right] , b _ { 2 } ^ { \dagger } } \right] = n B _ { n } ^ { \dagger } , \quad \left[ \begin{array}{c} { \left[ B _ { n } ^ { \neg } , b _ { 2 } ^ { \dagger } \right] , b _ { 2 } ^ { \neg } } \right] = n B


In [ ]:
model.push_to_hub("HF_UserName", token = "HF_TOKEN")

README.md:   0%|          | 0.00/543 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/Pt-kunal-mishra/gemma_4_lora


In [25]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%                                        48.5%###########################                                     52.3%                                  52.4%##############                                  55.7%#################                             62.8%########################                             62.9%#################                             63.4%#################################                         69.3%#################################################                    76.2%######################                   77.0%#########################################                 80.0%########################################      94.8%###################################     96.3%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding 

In [24]:
print("Installing zstd, please re-run the Ollama installation cell (NUxcyP_UfeLl) after this.")
!sudo apt-get install -y zstd

Installing zstd, please re-run the Ollama installation cell (NUxcyP_UfeLl) after this.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 133 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 2s (379 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
Selecting previously unselected package zstd.
(Reading database ... 124626 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3bu

In [ ]:
push=model.push_to_hub_gguf(
        "HF_userName", # Change hf to your username!
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "HF_TOKEN", # Get a token at https://huggingface.co/settings/tokens
    )